# Team Logistics — Beam Design with ANSYS and AI Agents

Simply supported steel beam, L = 1000 mm, central load P = 10 kN. Minimize mass subject to
midspan deflection ≤ 1.0 mm and nominal midspan bottom-surface von Mises stress ≤ 120 MPa,
with 20 ≤ b ≤ 100 mm and 40 ≤ h ≤ 120 mm.

This notebook reproduces every table and plot from the saved CSV files. Cells that start ANSYS
run only when `RUN_ANSYS = True` (requires ANSYS 2026 R1 and `C:\beam_project\beam.wbpj`).
Cells that call the AI model need `OPENAI_API_KEY` in the environment (never stored in files).

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

import beam_tools as bt
from beam_model import evaluate_beam

RUN_ANSYS = False      # set True to re-run ANSYS steps (slow: about 4 min per design)
RUN_AGENT = False      # set True to re-run the agent workflows (needs OPENAI_API_KEY and ANSYS)
HERE = Path('.')

## Task 1 — ANSYS model verification

Model: 3D solid (SpaceClaim `beam_geometry_spaceclaim.py`), X = span, Y = height, Z = width.
Pin support (X, Y, Z fixed) on the left bottom edge, roller (Y fixed) on the right bottom edge,
10 kN downward on a 10 mm wide strip at midspan (top face). Structural Steel (E = 200 GPa,
ν = 0.3, ρ = 7850 kg/m³). Nominal stress = average von Mises over bottom-surface nodes with
495 ≤ X ≤ 505 mm and |Z| ≤ 5 mm (`NS_NOMINAL_PROBE`).

### Mesh refinement at the start design (50 × 80 mm)

In [ ]:
ms = pd.read_csv('beam_mesh_study.csv')
ms['defl_change_pct'] = ms.deflection_mm.pct_change().abs() * 100
ms['stress_change_pct'] = ms.nominal_stress_mpa.pct_change().abs() * 100
ms['reaction_error_pct'] = (ms.pin_ry_n + ms.roller_ry_n - 10000).abs() / 10000 * 100
ms

Between the two finest meshes the deflection changes by less than 0.1 % (limit 2 %) and the
nominal stress by less than 0.1 % (limit 5 %). The reactions balance the load to 0.00 % (limit 1 %).
The global peak stress keeps growing with refinement (49 → 72 → 99 MPa): it is a singularity
at the support edges and the load strip, so it is not used as the constraint stress.

### Python vs ANSYS at the five required designs (8 mm search mesh)

In [ ]:
if RUN_ANSYS:
    %run ansys_five_designs.py
fd = pd.read_csv('team_logistics_five_design_comparison.csv')
cols = ['width_mm','height_mm',
        'python_deflection_mm','ansys_deflection_mm','signed_diff_deflection_mm','abs_pct_diff_deflection_mm',
        'python_nominal_stress_mpa','ansys_nominal_stress_mpa','signed_diff_nominal_stress_mpa','abs_pct_diff_nominal_stress_mpa',
        'python_feasible','ansys_feasible','ansys_nodes']
fd[cols].round(4)

Signed difference = Python − ANSYS; percentage relative to ANSYS. Python under-predicts the
deflection, and the gap grows with h/L (0.5 % at h = 40 mm, about 7 % at h = 120 mm) because
Euler–Bernoulli theory ignores shear deformation. Python over-predicts the nominal stress by
1–2.5 % because the 10 mm strip spreads the load and slightly lowers the midspan moment.

## Task 2 — Python integration

In [ ]:
print(bt.evaluate_python(50, 80))

In [ ]:
if RUN_ANSYS:
    from ansys_tools import evaluate_ansys, start_ansys, stop_ansys
    print('startup_s', start_ansys())
    for b, h in [(50, 80), (60, 100), (50, 80)]:   # two geometry changes, then a cache hit
        r = evaluate_ansys(b, h)
        print(b, h, r['status'], r['mass_kg'], r['deflection_mm'], r['nominal_stress_mpa'], 'cache_hit', r['cache_hit'])
    stop_ansys()
else:
    print(open('logs/smoke_test_log.txt').read() if Path('logs/smoke_test_log.txt').exists() else 'see README: smoke test log')

Failure handling (runs without ANSYS; a simulated crash must give no numbers, no cache entry, no Python fallback):

In [ ]:
%run ansys_failure_test.py

## Task 3 — Agent policy

One OpenAI model (`gpt-4.1`) with one system prompt for all workflows. Each reply must be JSON with
`width_mm`, `height_mm`, `tool`, `reason`. The Python controller (`beam_agent.py`) validates the
JSON, the bounds, the tool permission of the workflow and the ANSYS budget; a rejected proposal
is logged and still uses one iteration. One ANSYS evaluation is always reserved for the final
recommendation. No numerical optimizer generates candidates.

In [ ]:
import beam_agent as ba
print(ba.SYSTEM_PROMPT)
pd.DataFrame(ba.WORKFLOWS).T

In [ ]:
wfC = dict(ba.WORKFLOWS['C'], key='C')
for text in ['{"width_mm": 20, "height_mm": 87, "tool": "both", "reason": "x"}',
             '{"width_mm": 150, "height_mm": 87, "tool": "python", "reason": "x"}',
             '{"width_mm": 20, "height_mm": 87, "tool": "ansys", "reason": "x"}',
             'not json']:
    left = 0 if '"ansys"' in text else 3
    print(text[:60].ljust(62), '->', ba.validate(text, wfC, left)[1] or 'valid')

## Task 4 — Workflow comparison

In [ ]:
if RUN_AGENT:
    %run beam_agent.py --workflow all
res = pd.read_csv('team_logistics_beam_results.csv')
print(len(res), 'rows (one per tool execution; invalid proposals kept)')
res[['workflow','iteration','phase','row_type','proposed_tool','width_mm','height_mm','model_source',
     'mass_kg','deflection_mm','nominal_stress_mpa','feasible','cache_hit','elapsed_s','rejection_reason']]

In [ ]:
%run beam_compare.py
display(Image('team_logistics_plot_progress.png'))
display(Image('team_logistics_plot_design_space.png'))

### Mesh check of the final designs

In [ ]:
if RUN_ANSYS:
    %run ansys_final_mesh_check.py
pd.read_csv('team_logistics_final_mesh_check.csv') if Path('team_logistics_final_mesh_check.csv').exists() else 'run ansys_final_mesh_check.py'

### Was adaptive selection worth it?

| Workflow | Final design | Mass | ANSYS-feasible | New ANSYS solves | ANSYS time |
|---|---|---|---|---|---|
| A | 20 × 86.5 | 13.58 kg | yes (0.9967 mm) | 9 | 35 min |
| B | 20 × 86.0 | 13.50 kg | **no** (1.0138 mm) | 1 | 4 min |
| C | 21 × 85.0 | 14.01 kg | yes (0.9991 mm) | 5 | 17 min |

**Yes against B, partly against A.** B's Python-only answer violates the deflection limit in ANSYS, because the
shear-deformation gap (about 3 %) is larger than B's Python margin (1.7 %). C reached an ANSYS-verified design with half
of A's ANSYS time. Its final mass is 3.2 % above A's, but it had already verified 20 × 87 mm (13.66 kg, only 0.6 % above A)
in iteration 4: the loss came from the agent's final choice, not from the adaptive tool policy. All final designs are
mesh-converged (< 0.1 % change between 8 and 6 mm meshes); C's margin shrinks to 0.0001 mm on the 6 mm mesh.